In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, root_scalar
from sklearn.metrics import r2_score

In [23]:
def complexityN0(n_data, t_dijkstra, std_dijkstra, t_bmssp, std_bmssp):
    def model_dijkstra(n, c):
        return c * (n * np.log2(n))

    def model_bmssp(n, c):
        return c * (n * np.power(np.log2(n), 2/3))
    
    params_d, _ = curve_fit(model_dijkstra, n_data, t_dijkstra, bounds=(0, np.inf), sigma=std_dijkstra, absolute_sigma=True)
    params_b, _ = curve_fit(model_bmssp, n_data, t_bmssp, bounds=(0, np.inf), sigma=std_bmssp, absolute_sigma=True)

    c_d = params_d[0]
    c_b = params_b[0]

    if c_d == 0 or c_b == 0:
        return -1
    
    log_n_intersect = np.power(c_b / c_d, 3).item()

    if log_n_intersect < np.log2(n_data[-1]):
        log_n_intersect = -1
    
    r2_d = r2_score(t_dijkstra, model_dijkstra(n_data, c_d))
    r2_b = r2_score(t_bmssp, model_bmssp(n_data, c_b))

    return {    
        'intersect_power10': round(log_n_intersect * np.log10(2), 3), # convert to power10
        'dijkstra': {
            "r2": round(r2_d, 3),
            "c": c_d.item()
        },
        'bmssp': {
            "r2": round(r2_b, 3),
            "c": c_b.item()
        }
    }

In [24]:
import pandas as pd

results_list = []
for graph_type in ['H3', 'D3', 'USA']:
    df = pd.read_csv(f'../outputs/results_{graph_type}/performance_summary_table.csv')
    n_data = df['Número de Vértices'].values
    t_dijkstra = df['Tempo Dijkstra (ms)'].values
    std_dijkstra = df['Desvio Dijkstra (ms)'].values
    t_bmssp = df['Tempo BMSPP (ms)'].values
    std_bmssp = df['Desvio BMSPP (ms)'].values
    
    res_comp = complexityN0(n_data, t_dijkstra, std_dijkstra, t_bmssp, std_bmssp)

    results_list.append({
        'Graph': graph_type,
        'C Dijkstra': f"{res_comp['dijkstra']['c']:.2e}",
        'R^2 Dijkstra': f"{res_comp['dijkstra']['r2']:.3f}",
        'C BMSSP': f"{res_comp['bmssp']['c']:.2e}",
        'R^2 BMSSP': f"{res_comp['bmssp']['r2']:.3f}",
        'C Ratio': f"{float(res_comp['bmssp']['c']) / float(res_comp['dijkstra']['c']):.2f}",
        'n0_log10': res_comp['intersect_power10'],
    })

for grid_type in ['GridR', 'GridED']:
    for grid_variant in ['S', 'R']:
        graph_name = f"{grid_variant}{grid_type}"
        df = pd.read_csv(f'../outputs/results_{grid_type}/performance_summary_table.csv')
        df = df[df['Graph File'].str.contains(f'{graph_name}')].copy()
        n_data = df['Número de Vértices'].values
        t_dijkstra = df['Tempo Dijkstra (ms)'].values
        std_dijkstra = df['Desvio Dijkstra (ms)'].values
        t_bmssp = df['Tempo BMSPP (ms)'].values
        std_bmssp = df['Desvio BMSPP (ms)'].values
        
        res_comp = complexityN0(n_data, t_dijkstra, std_dijkstra, t_bmssp, std_bmssp)

        results_list.append({
            'Graph': graph_name,
            'C Dijkstra': f"{res_comp['dijkstra']['c']:.2e}",
            'R^2 Dijkstra': f"{res_comp['dijkstra']['r2']:.3f}",
            'C BMSSP': f"{res_comp['bmssp']['c']:.2e}",
            'R^2 BMSSP': f"{res_comp['bmssp']['r2']:.3f}",
            'C Ratio': f"{float(res_comp['bmssp']['c']) / float(res_comp['dijkstra']['c']):.2f}",
            'n0_log10': res_comp['intersect_power10'],
        })

summary_df = pd.DataFrame(results_list)
display(summary_df)

,Graph,C Dijkstra,R^2 Dijkstra,C BMSSP,R^2 BMSSP,C Ratio,n0_log10
0,H3,2.25e-05,0.979,1.96e-04,0.887,8.74,200.746
1,D3,2.46e-05,0.997,2.14e-04,0.936,8.69,197.564
2,USA,7.62e-06,0.975,7.63e-05,0.919,10.02,302.814
3,SGridR,1.52e-05,0.977,1.49e-04,0.988,9.85,287.512
4,RGridR,9.25e-06,0.997,7.05e-05,0.887,7.63,133.614
5,SGridED,6.63e-06,0.999,1.17e-04,0.999,17.72,1675.365
6,RGridED,6.07e-06,0.987,1.14e-04,0.999,18.77,1989.293


In [26]:
def format_scientific(value):
    if value == 0:
        return "0"
    exponent = int(np.floor(np.log10(abs(value))))
    mantissa = value / (10 ** exponent)
    return f"{mantissa:.2f} \\times 10^{{{exponent}}}"

latex_output = r"""\begin{table}[!htpb]
    \centering
    \caption{Complexity analysis: Dijkstra vs BMSSP-WC}
    \begin{tabular}{lccccccc}
        \hline
        \textbf{Graph} & \textbf{C Dijkstra} & $\mathbf{R^2}$ \textbf{Dijkstra} & \textbf{C BMSSP} & $\mathbf{R^2}$ \textbf{BMSSP} & \textbf{C Ratio} & \textbf{n0} \\
        \hline
"""

for _, row in summary_df.iterrows():
    c_dijkstra = row['C Dijkstra']
    c_bmssp = row['C BMSSP']
    
    c_dijkstra_formatted = format_scientific(float(c_dijkstra))
    c_bmssp_formatted = format_scientific(float(c_bmssp))
    
    latex_output += f"        {row['Graph']} & ${c_dijkstra_formatted}$ & ${row['R^2 Dijkstra']}$ & ${c_bmssp_formatted}$ & ${row['R^2 BMSSP']}$ & ${row['C Ratio']}$ & $10^{{{row['n0_log10']}}}$ \\\\\n"

latex_output += r"""        \hline
    \end{tabular}
    \label{tab:complexity_analysis}
\end{table}
"""

print(latex_output)

\begin{table}[!htpb]
    \centering
    \caption{Complexity analysis: Dijkstra vs BMSSP-WC}
    \begin{tabular}{lccccccc}
        \hline
        \textbf{Graph} & \textbf{C Dijkstra} & $\mathbf{R^2}$ \textbf{Dijkstra} & \textbf{C BMSSP} & $\mathbf{R^2}$ \textbf{BMSSP} & \textbf{C Ratio} & \textbf{n0} \\
        \hline
        H3 & $2.25 \times 10^{-5}$ & $0.979$ & $1.96 \times 10^{-4}$ & $0.887$ & $8.74$ & $10^{200.746}$ \\
        D3 & $2.46 \times 10^{-5}$ & $0.997$ & $2.14 \times 10^{-4}$ & $0.936$ & $8.69$ & $10^{197.564}$ \\
        USA & $7.62 \times 10^{-6}$ & $0.975$ & $7.63 \times 10^{-5}$ & $0.919$ & $10.02$ & $10^{302.814}$ \\
        SGridR & $1.52 \times 10^{-5}$ & $0.977$ & $1.49 \times 10^{-4}$ & $0.988$ & $9.85$ & $10^{287.512}$ \\
        RGridR & $9.25 \times 10^{-6}$ & $0.997$ & $7.05 \times 10^{-5}$ & $0.887$ & $7.63$ & $10^{133.614}$ \\
        SGridED & $6.63 \times 10^{-6}$ & $0.999$ & $1.17 \times 10^{-4}$ & $0.999$ & $17.72$ & $10^{1675.365}$ \\
        RGridED 